# SARIMAX — NY Daily Electricity Demand (3-day forecast)

**Design choice (diverges from handoff, read this):**
The XGBoost plan uses 3 separate models on `target_day1/2/3`. For SARIMAX that is **wrong** —
the AR/MA terms on a `target_dayN` series would consume *future* demand values (leakage).

Idiomatic + leakage-free SARIMAX:
- **endog = `demand`** (the daily series itself).
- Weekly seasonality via seasonal order `m=7`.
- Annual seasonality via **Fourier exog terms** (m=365 is infeasible for SARIMAX).
- **exog = weather + economic + holiday + Fourier/trend only.**
  Excluded: demand lags & rolling (captured by AR), generation/interchange (unknown at
  forecast time → leakage), `demand_forecast` (EIA leakage), raw temporal ints (replaced by Fourier + seasonal m=7).
- Per-horizon Day+1/+2/+3 metrics produced via **walk-forward 3-step forecasts** → satisfies project reporting.

Switch to the literal 3-target approach only if you need strict parity with XGBoost; tell me and I'll rewrite.

In [1]:
# Optional installs (uncomment on a fresh env)
#%pip install statsmodels pmdarima mlflow python-dotenv pyarrow scikit-learn

In [2]:
import warnings, os, json, itertools, pickle
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [18]:
TRACKING_OK = False
from dotenv import load_dotenv
load_dotenv(".env")
try:
    from config import *
    PROC = globals().get("PROCESSED_DIR", "../data/processed")
    MODELS = globals().get("MODELS_DIR", "../models")
except Exception:
    PROC, MODELS = "../data/processed", "../models"
os.makedirs(MODELS, exist_ok=True)
os.makedirs("../reports", exist_ok=True)
try:
    import mlflow
    uri = os.getenv("MLFLOW_TRACKING_URI")
    if uri:
        os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("DAGSHUB_USER_NAME")
        os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("DAGSHUB_TOKEN")
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("sarimax")
        TRACKING_OK = True
        print("MLflow ->", uri)
    else:
        print("No MLFLOW_TRACKING_URI; skipping remote logging.")
except Exception as e:
    print("MLflow unavailable:", e)

In [4]:
# --- Load ---
def load(name):
    p = os.path.join(PROC, name)
    df = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv(p.replace(".parquet", ".csv"))
    date_col = "date" if "date" in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df.index = pd.DatetimeIndex(df.index).tz_localize(None)
    return df.asfreq("D")            # 0 missing days per handoff

train = load("features_selected_train.parquet")
test  = load("features_selected_test.parquet")
print(train.shape, test.shape)
print("missing demand:", train["demand"].isna().sum(), test["demand"].isna().sum())
train.columns.tolist()

In [5]:
# --- Build exogenous matrix ---
# Drop: endog, targets, demand-derived (AR handles), generation/interchange (future-unknown),
# raw temporal ints (seasonal m=7 + Fourier handle), is_weekend (== m=7).
DROP_EXACT = {"demand","demand_forecast","target_day1","target_day2","target_day3",
              "net_generation","total_interchange","is_weekend",
              "year","month","day_of_month","day_of_week","week_of_year","quarter"}
DROP_PREFIX = ("ng_","demand_lag","demand_roll")

def raw_exog(df):
    num = df.select_dtypes("number")
    keep = [c for c in num.columns
            if c not in DROP_EXACT and not c.startswith(DROP_PREFIX)]
    return num[keep]

ex_tr = raw_exog(train)
ex_te = raw_exog(test)
print("raw exog (%d):" % ex_tr.shape[1], ex_tr.columns.tolist())

In [6]:
# --- Prune collinearity ---
# Explicit (handoff): keep temperature_2m, drop apparent_temperature / dew_point variants.
def name_has(c, *keys): return any(k in c.lower() for k in keys)
explicit = [c for c in ex_tr.columns if name_has(c,"apparent_temperature","dew_point")]
ex_tr = ex_tr.drop(columns=explicit); ex_te = ex_te.drop(columns=explicit)

# Greedy |r|>0.95 prune on remaining.
corr = ex_tr.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
ex_tr = ex_tr.drop(columns=drop_corr); ex_te = ex_te.drop(columns=drop_corr)
print("dropped explicit:", explicit)
print("dropped corr>0.95:", drop_corr)
print("exog after prune (%d):" % ex_tr.shape[1], ex_tr.columns.tolist())

In [7]:
# --- Add annual Fourier terms + linear trend (defined for any date) ---
N_FOURIER = 3
T0 = train.index.min()
def time_feats(idx):
    doy = idx.dayofyear.values.astype(float)
    out = {}
    for k in range(1, N_FOURIER+1):
        out[f"sin{k}"] = np.sin(2*np.pi*k*doy/365.25)
        out[f"cos{k}"] = np.cos(2*np.pi*k*doy/365.25)
    out["trend"] = (idx - T0).days.values.astype(float)
    return pd.DataFrame(out, index=idx)

ex_tr = pd.concat([ex_tr, time_feats(ex_tr.index)], axis=1)
ex_te = pd.concat([ex_te, time_feats(ex_te.index)], axis=1)
ex_tr = ex_tr.ffill().bfill(); ex_te = ex_te.ffill().bfill()
EXOG_COLS = ex_tr.columns.tolist()
print("final exog (%d):" % len(EXOG_COLS), EXOG_COLS)

In [8]:
# --- Scale exog (fit on train; aids convergence) ---
scaler = StandardScaler().fit(ex_tr.values)
Xtr = pd.DataFrame(scaler.transform(ex_tr.values), index=ex_tr.index, columns=EXOG_COLS)
Xte = pd.DataFrame(scaler.transform(ex_te.values), index=ex_te.index, columns=EXOG_COLS)
ytr = train["demand"].astype(float)
yte = test["demand"].astype(float)

In [9]:
# --- Stationarity (ADF) ---
def adf(s, label):
    r = adfuller(s.dropna(), autolag="AIC")
    print(f"{label:18s} ADF={r[0]:.3f}  p={r[1]:.4f}  -> {'stationary' if r[1]<0.05 else 'NON-stationary'}")
    return r[1]
p0 = adf(ytr, "demand")
p1 = adf(ytr.diff(), "diff(1)")
p7 = adf(ytr.diff(7), "seasonal diff(7)")
d_hint = 0 if p0 < 0.05 else 1
print("d hint:", d_hint)

In [10]:
# --- Order selection: auto_arima if available, else AIC grid ---
order = seasonal_order = None
try:
    import pmdarima as pm
    am = pm.auto_arima(ytr, X=Xtr, seasonal=True, m=7,
                       d=None, D=None, max_p=3, max_q=3, max_P=2, max_Q=2,
                       stepwise=True, suppress_warnings=True, error_action="ignore",
                       information_criterion="aic", trace=True)
    order, seasonal_order = am.order, am.seasonal_order
    print("auto_arima ->", order, seasonal_order)
except Exception as e:
    print("pmdarima unavailable -> AIC grid.", e)
    best = (np.inf, None, None)
    pdq = list(itertools.product(range(3), [d_hint], range(3)))
    PDQ = [(P, 1, Q, 7) for P in range(2) for Q in range(2)]
    for o in pdq:
        for so in PDQ:
            try:
                m = SARIMAX(ytr, exog=Xtr, order=o, seasonal_order=so,
                            enforce_stationarity=False, enforce_invertibility=False)
                aic = m.fit(disp=False, maxiter=50, method="lbfgs").aic
                if aic < best[0]: best = (aic, o, so)
            except Exception:
                continue
    _, order, seasonal_order = best
    print("grid best ->", order, seasonal_order, "AIC", best[0])

In [11]:
# --- Fit final model ---
model = SARIMAX(ytr, exog=Xtr, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False)
res = model.fit(disp=False, maxiter=300, method="lbfgs")
print(res.summary())

In [13]:
# --- Walk-forward 3-step forecast over test (per-horizon, leakage-free) ---
rows = []
wf = res
n = len(yte)
for o in range(n):
    h = min(3, n - o)
    fc = wf.get_forecast(steps=h, exog=Xte.iloc[o:o+h])
    mean = np.asarray(fc.predicted_mean)
    for k in range(h):
        rows.append({"date": yte.index[o+k], "horizon": k+1,
                     "yhat": mean[k], "y": yte.iloc[o+k]})
    wf = wf.append(endog=yte.iloc[o:o+1], exog=Xte.iloc[o:o+1], refit=False)

pred = pd.DataFrame(rows)
def metrics(g):
    y, yh = g["y"].values, g["yhat"].values
    rmse = np.sqrt(mean_squared_error(y, yh))
    mae = mean_absolute_error(y, yh)
    mape = np.mean(np.abs((y - yh) / y)) * 100
    return pd.Series({"RMSE": rmse, "MAE": mae, "MAPE_%": mape, "n": len(g)})
summary = pred.groupby("horizon").apply(metrics)
print(summary.round(2))

In [14]:
# --- Plot Day+1 actual vs predicted ---
h1 = pred[pred.horizon == 1].set_index("date")
plt.figure(figsize=(13,4))
plt.plot(h1.index, h1["y"], label="actual", lw=1)
plt.plot(h1.index, h1["yhat"], label="SARIMAX Day+1", lw=1, alpha=.8)
plt.title("SARIMAX — Day+1 demand (test)"); plt.ylabel("MWh"); plt.legend(); plt.tight_layout()
plt.savefig("../reports/sarimax_day1_test.png", dpi=120); plt.close()
print("saved ../reports/sarimax_day1_test.png")

In [ ]:
# --- Persist model + scaler + exog spec ---
res.save(os.path.join(MODELS, "sarimax_demand.pkl"))
with open(os.path.join(MODELS, "sarimax_artifacts.pkl"), "wb") as f:
    pickle.dump({"scaler":scaler, "exog_cols":EXOG_COLS, "order":order,
                 "seasonal_order":seasonal_order, "n_fourier":N_FOURIER,
                 "t0":T0}, f)
print("saved model + scaler to", MODELS)

In [20]:
# --- MLflow logging ---
if TRACKING_OK:
    with mlflow.start_run(run_name="sarimax_demand_m7"):
        mlflow.log_params({"model":"SARIMAX","endog":"demand",
                           "order":str(order),"seasonal_order":str(seasonal_order),
                           "n_exog":len(EXOG_COLS),"n_fourier":N_FOURIER})
        mlflow.log_metric("aic", float(res.aic))
        for hh in summary.index:
            for m in ["RMSE","MAE","MAPE_%"]:
                mlflow.log_metric(f"{m}_day{hh}".replace('%','pct'), float(summary.loc[hh, m]))
        summary.to_csv("../reports/sarimax_metrics.csv"); mlflow.log_artifact("../reports/sarimax_metrics.csv")
        mlflow.log_artifact("../reports/sarimax_day1_test.png")
        mlflow.log_artifact(os.path.join(MODELS, "sarimax_demand.pkl"))
        mlflow.log_artifact(os.path.join(MODELS, "sarimax_artifacts.pkl"))
    print("logged to MLflow")
else:
    summary.to_csv("../reports/sarimax_metrics.csv")
    print("saved metrics locally")